# RAMEC — Revisión Automática Multimodal de Entregables de Construcción

Notebook de **ejecución en Google Colab**: clona el repositorio, instala las dependencias,
descarga los pesos del modelo y ejecuta las validaciones.

**Antes de empezar:**

(Opcional) Activa GPU en *Entorno de ejecución → Cambiar tipo de entorno → GPU*.
   La inferencia también funciona en CPU.

## 1. Configuración

In [1]:
REPO = "BRIDERI/Ramec"
TAG  = "v1.0"              # etiqueta del Release que contiene los pesos best.pt

## 2. (Opcional) Verificar GPU

In [ ]:
!nvidia-smi || echo "Sin GPU: se usará CPU (la inferencia funciona igual)."

/bin/bash: line 1: nvidia-smi: command not found
Sin GPU: se usará CPU (la inferencia funciona igual).


## 3. Clonar el repositorio

In [2]:
%cd /content
!rm -rf /content/ramec
!git clone https://github.com/{REPO}.git /content/ramec
%cd /content/ramec
!grep -n "Firma_Elaboró" src/report.py
!grep -n "analizar_firmas_control" src/extract.py
!grep -n "VAL_CTRL" src/infer.py

/content
Cloning into '/content/ramec'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 100 (delta 46), reused 32 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 68.56 KiB | 1.40 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/ramec
158:        "Firma_Elaboró": _sn(firmas[0]),
186:                "Firma_Elaboró", "Firma_Revisó", "Firma_Aprobó_1", "Firma_Aprobó_2",
674:def analizar_firmas_control(crop, filas_esperadas=None):
40:VAL_CTRL = C.NAME_TO_ID["validacion_profesional_hoja_control"]  # 9
161:        if VAL_CTRL in boxes:
162:            firma_crop = EX.crop_box(img, boxes[VAL_CTRL][0])


## 4. Dependencias del sistema (Tesseract + Poppler)
No se instalan con pip; el idioma `spa` es necesario para leer texto en español.

In [3]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Selecting previously unselected package tesseract-ocr-spa.
Preparing to unpa

## 5. Dependencias de Python

In [4]:
!pip -q install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 539.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 101.9 MB/s eta 0:00:00


## 6. Descargar los pesos del modelo (desde el Release)
Deja `models/planos/best.pt` y `models/documentos/best.pt`.

In [5]:
!REPO={REPO} TAG={TAG} bash scripts/download_models.sh

Descargando pesos desde https://github.com/BRIDERI/Ramec/releases/download/v1.0 ...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 38.8M  100 38.8M    0     0  16.6M      0  0:00:02  0:00:02 --:--:-- 23.1M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 38.7M  100 38.7M    0     0  10.0M      0  0:00:03  0:00:03 --:--:-- 12.4M
Listo. Pesos descargados:
-rw-r--r-- 1 root root 39M Jul  9 16:15 models/documentos/best.pt
-rw-r--r-- 1 root root 39M Jul  9 16:14 models/planos/best.pt


## 7. Subir los PDFs a revisar
Sube uno o más PDFs (planos y/o documentos). Se guardan en la carpeta `pdfs/`.

In [17]:
import os
os.makedirs("pdfs", exist_ok=True)
from google.colab import files
print("Selecciona uno o más PDFs...")
subidos = files.upload()
for nombre in subidos:
    os.replace(nombre, os.path.join("pdfs", nombre))
!ls -lh pdfs

Selecciona uno o más PDFs...


Saving AVP-SAV-9000-D-INF-CYT-ING-000001.pdf to AVP-SAV-9000-D-INF-CYT-ING-000001.pdf
total 191M
-rw-r--r-- 1 root root 112M Jul  9 16:24 AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf
-rw-r--r-- 1 root root  79M Jul  9 20:21 AVP-SAV-9000-D-INF-CYT-ING-000001.pdf


*Alternativa:* en vez de subir archivos, puedes montar tu Google Drive y apuntar a una
carpeta con PDFs (descomenta y ajusta la ruta).

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p pdfs && cp /content/drive/MyDrive/ruta/a/tus/pdfs/*.pdf pdfs/

## 8. Ejecutar la validación

In [18]:
!python src/infer.py --pdfs pdfs --salida outputs/Reporte_validacion.xlsx

Listo: outputs/Reporte_validacion.xlsx
  planos=0 documentos=2 validacion_profesional=2 filas estandar=2


## 9. Ver y descargar el reporte
Se genera un Excel con seis hojas de verificación.

In [ ]:
import pandas as pd
xls = pd.ExcelFile("outputs/Reporte_validacion.xlsx")
print("Hojas:", xls.sheet_names)
display(pd.read_excel(xls, "VALIDACION_PROFESIONAL"))

from google.colab import files
files.download("outputs/Reporte_validacion.xlsx")

Hojas: ['ESTANDAR NOMENCLATURA', 'COMPATIBILIDAD_DOCUMENTO', 'COHERENCIA_DOCUMENTO', 'CONTROL_CAMBIOS_DOC', 'VALIDACION_PROFESIONAL']


,Ruta,Archivo,Tipo,Elaboró,Revisó,Aprobó_1,Aprobó_2,Firma_Elaboró,Firma_Revisó,Firma_Aprobó_1,Firma_Aprobó_2,Fecha_validacion,Responsables_completos,Firmas_completas,Fecha_coincide,Validacion_profesional
0,pdfs/AVP-SAV-3000-D-EST-TRA-ING-000001.pdf,AVP-SAV-3000-D-EST-TRA-ING-000001.pdf,DOCUMENTO,Rosendo Torrens Pujadas,Eugenia Marina Acero Carrión,Armando González González,Miguel ángel Ordóñez Gutiérrez,SI,SI,SI,SI,12/06/2026,SI,SI,SI,SI
1,pdfs/AVP-SAV-3000-D-MDE-SGA-ING-000001.pdf,AVP-SAV-3000-D-MDE-SGA-ING-000001.pdf,DOCUMENTO,Eliseo Palomares,Mayra Gómez Sandoval,Armando González González,Miguel Ángel Ordóñez Gutiérrez,SI,SI,SI,SI,12/06/2026,SI,SI,SI,SI


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## (Opcional) Reentrenar los modelos
Requiere el **dataset anotado en CVAT** en `data/raw/` (no incluido en el repo por
confidencialidad/tamaño). Con tu dataset disponible, descomenta:

In [ ]:
# !python src/convert.py --planos data/raw/planos --documentos data/raw/documentos --val-frac 0.15
# !python src/train.py --task both     # genera models/<task>/best.pt

In [19]:
%cd /content/ramec

from pathlib import Path

print("Ruta actual:")
!pwd

print("\nArchivos principales:")
!ls -lh src configs models pdfs outputs 2>/dev/null

print("\nModelos:")
!find models -type f -name "*.pt"

print("\nPDFs cargados:")
!find pdfs -type f -name "*.pdf"

print("\nScripts clave:")
!ls -lh src/extract.py src/infer.py src/report.py configs/classes.py

/content/ramec
Ruta actual:
/content/ramec

Archivos principales:
configs:
total 16K
-rw-r--r-- 1 root root 2.7K Jul  9 16:09 classes.py
-rw-r--r-- 1 root root  959 Jul  9 16:09 documentos.yaml
-rw-r--r-- 1 root root  939 Jul  9 16:09 planos.yaml
drwxr-xr-x 2 root root 4.0K Jul  9 16:24 __pycache__

models:
total 8.0K
drwxr-xr-x 2 root root 4.0K Jul  9 16:14 documentos
drwxr-xr-x 2 root root 4.0K Jul  9 16:14 planos

outputs:
total 32K
-rw-r--r-- 1 root root 8.5K Jul  9 20:23 Reporte_validacion.xlsx
drwxr-xr-x 2 root root 4.0K Jul  9 19:40 validacion_logos
drwxr-xr-x 2 root root 4.0K Jul  9 16:36 validacion_paginas
-rw-r--r-- 1 root root 9.4K Jul  9 16:36 VALIDACION_PAGINAS.csv

pdfs:
total 191M
-rw-r--r-- 1 root root 112M Jul  9 16:24 AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf
-rw-r--r-- 1 root root  79M Jul  9 20:21 AVP-SAV-9000-D-INF-CYT-ING-000001.pdf

src:
total 76K
-rw-r--r-- 1 root root 6.2K Jul  9 16:09 convert.py
-rw-r--r-- 1 root root  25K Jul  9 16:09 extract.py
-rw-r--r-- 1 root

In [31]:
%cd /content/ramec

# Instalación por si Colab no tiene OCR activo
!apt-get -qq update >/dev/null
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa >/dev/null
!pip -q install ultralytics pymupdf pytesseract openpyxl pandas 'Pillow==9.5.0'

from pathlib import Path
import re
import fitz
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO
import pytesseract

# =========================
# CONFIGURACIÓN
# =========================
PDF_DIR = Path("pdfs")
OUT_DIR = Path("outputs/validacion_paginas")
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPORTE_XLSX = Path("outputs/Reporte_validacion.xlsx")
MODELO_DOC = Path("models/documentos/best.pt")

# Para prueba rápida usa 10 páginas.
# Cuando veas que funciona, cambia a None para procesar todo el PDF.
MAX_PAGES = 10

CONF = 0.25
DPI = 220
IMGSZ = 1280

model = YOLO(str(MODELO_DOC))
names = model.names

print("Clases del modelo:")
print(names)

# Clases que nos interesan para validación por página
CLASES_LOGO = {
    "logo_entidades_paginas",
    "logo_entidades_caratula"
}

CLASES_SELLO = {
    "firmas_aprobacion_paginas"
}

# Define sets for page type classification based on model classes
CLASES_CARATULA_IDENTIFIERS = {
    "proyecto_caratula", "titulo_documento_caratula", "num_documento_caratula",
    "fecha_caratula", "logo_entidades_caratula"
}
CLASES_HOJA_CONTROL_IDENTIFIERS = {
    "validacion_profesional_hoja_control", "num_documento_hoja_control",
    "titulo_documento_hoja_control", "fecha_aprobacion_hoja_control",
    "responsables_hoja_control", "num_revision_hoja_control"
}


# =========================
# FUNCIONES AUXILIARES
# =========================
def limpiar_texto(txt):
    txt = txt.replace("\x0c", " ")
    txt = re.sub(r"[ \t]+", " ", txt)
    txt = re.sub(r"\n{2,}", "\n", txt)
    return txt.strip()


def preprocesar_ocr(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    return gray


def ocr_mejor_rotacion(img_rgb):
    """
    Prueba varias rotaciones y se queda con el texto más útil.
    Para tus sellos laterales normalmente funciona mejor rotación 270 + psm 6.
    """
    candidatos = []

    for rot in [0, 90, 180, 270]:
        pil = Image.fromarray(img_rgb).rotate(rot, expand=True)
        arr = np.array(pil)
        proc = preprocesar_ocr(arr)

        for psm in [6, 11, 12]:
            config = f"--oem 3 --psm {psm}"
            txt = pytesseract.image_to_string(proc, lang="spa+eng", config=config)
            txt = limpiar_texto(txt)

            score = len(re.sub(r"[^A-Za-zÁÉÍÓÚáéíóúÑñ0-9]", "", txt))

            # Premios si detecta cosas importantes
            if re.search(r"\bCIP\b|Reg.?\s*CIP", txt, re.IGNORECASE):
                score += 40
            if re.search(r"GERENTE|JEFE|ESPECIALISTA|PROYECTO|CONCESIONARIA", txt, re.IGNORECASE):
                score += 25

            candidatos.append((score, rot, psm, txt))

    candidatos.sort(reverse=True, key=lambda x: x[0])
    mejor = candidatos[0]
    return mejor[3], mejor[1], mejor[2]


def extraer_datos_sello(texto):
    lineas = [l.strip() for l in texto.splitlines() if l.strip()]
    texto_u = texto.upper()

    # CIP
    cip = ""
    m = re.search(r"(?:REG.?\s*)?CIP.?\s*(?:N[°º]?\s*)?(\d{3,8})", texto_u)
    if m:
        cip = m.group(1)

    # Cargo
    cargo = ""
    claves_cargo = [
        "GERENTE", "JEFE", "ESPECIALISTA", "COORDINADOR",
        "RESPONSABLE", "SUPERVISOR", "INGENIERO", "ARQUITECTO"
    ]
    for l in lineas:
        u = l.upper()
        if any(k in u for k in claves_cargo):
            cargo = l
            break

    # Nombre probable
    nombre = ""
    excluir = [
        "CIP", "REG", "GERENTE", "JEFE", "ESPECIALISTA",
        "COORDINADOR", "RESPONSABLE", "SUPERVISOR",
        "SOCIEDAD", "CONCESIONARIA", "PROYECTO", "SAC", "S.A.C"
    ]

    for l in lineas:
        u = re.sub(r"[^A-ZÁÉÍÓÚÑ ]", " ", l.upper())
        u = re.sub(r"\s+", " ", u).strip()

        if len(u.split()) >= 2 and not any(e in u for e in excluir):
            nombre = u.title()
            break

    return nombre, cargo, cip


def render_page(page, dpi=220):
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    return np.array(img)


# =========================
# PROCESAMIENTO
# =========================
rows = []

pdfs = sorted(PDF_DIR.glob("*.pdf"))

for pdf_path in pdfs:
    print(f"\nProcesando: {pdf_path.name}")

    doc = fitz.open(str(pdf_path))
    total_pages = len(doc)

    paginas_a_procesar = total_pages if MAX_PAGES is None else min(MAX_PAGES, total_pages)

    for page_idx in range(paginas_a_procesar):
        page_num = page_idx + 1
        img = render_page(doc[page_idx], dpi=DPI)

        result = model.predict(
            source=img,
            conf=CONF,
            imgsz=IMGSZ,
            verbose=False
        )[0]

        # Initialize lists to store detected info for logos and sellos
        detected_logos_text = [] # Store only the OCR'd text from logos
        detected_sellos_info = [] # Store all relevant info for sellos

        # Determine Tipo_página
        page_type = "Contenido" # Default value
        # Collect all detected class names for the current page
        detected_classes_on_page = {str(names[int(box.cls[0])]) for box in result.boxes}

        if any(cls_id in detected_classes_on_page for cls_id in CLASES_CARATULA_IDENTIFIERS):
            page_type = "Carátula"
        elif any(cls_id in detected_classes_on_page for cls_id in CLASES_HOJA_CONTROL_IDENTIFIERS):
            page_type = "Hoja de control"
        # If not Carátula or Hoja de control, it remains 'Contenido' by default,
        # or could be refined further if needed (e.g., 'Otros' if no 'contenido' detected)

        for box in result.boxes:
            cls_id = int(box.cls[0])
            cls_name = str(names[cls_id])
            conf = float(box.conf[0])

            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)

            crop = img[y1:y2, x1:x2]

            if cls_name in CLASES_LOGO:
                texto_ocr_logo, _, _ = ocr_mejor_rotacion(crop)
                detected_logos_text.append(texto_ocr_logo)
                # Opcional: guardar recorte de logo si es necesario para depuración
                # logo_crop_name = f"{pdf_path.stem}_pag_{page_num:03d}_logo_{len(detected_logos_text)}.png"
                # (OUT_DIR / logo_crop_name).parent.mkdir(parents=True, exist_ok=True)
                # Image.fromarray(crop).save(OUT_DIR / logo_crop_name)

            if cls_name in CLASES_SELLO:
                sello_crop_name = f"{pdf_path.stem}_pag_{page_num:03d}_sello_{len(detected_sellos_info)+1}.png"
                sello_crop_path = OUT_DIR / sello_crop_name
                Image.fromarray(crop).save(sello_crop_path)

                texto_ocr_sello, rotacion, psm = ocr_mejor_rotacion(crop)
                nombre, cargo, cip = extraer_datos_sello(texto_ocr_sello)

                detected_sellos_info.append({
                    "conf": conf,
                    "crop_path": str(sello_crop_path),
                    "texto_ocr": texto_ocr_sello,
                    "rotacion": rotacion,
                    "psm": psm,
                    "nombre": nombre,
                    "cargo": cargo,
                    "cip": cip
                })

        logo_detectado = "Sí" if detected_logos_text else "No"
        entidades_logo_detectadas_str = " | ".join(detected_logos_text) if detected_logos_text else ""

        # Ahora, construimos las filas en 'rows'
        if detected_sellos_info:
            # Si se detectaron sellos, una fila por cada sello
            for sello_data in detected_sellos_info:
                rows.append({
                    "Archivo": pdf_path.name,
                    "Página": page_num,
                    "Tipo_página": page_type,
                    "Logo_detectado": logo_detectado, # Usa la detección a nivel de página
                    "Entidades_logo_detectadas": entidades_logo_detectadas_str, # Nueva columna aquí
                    "Sello_detectado": "Sí",
                    "Nombre_sello": sello_data["nombre"],
                    "Cargo": sello_data["cargo"],
                    "CIP": sello_data["cip"],
                    "Confianza_sello": round(sello_data["conf"], 3),
                    "Rotación_OCR": sello_data["rotacion"],
                    "PSM_OCR": sello_data["psm"],
                    "Texto_OCR": sello_data["texto_ocr"],
                    "Recorte": sello_data["crop_path"],
                    "Validación_página": "Página con logo y sello detectado" if logo_detectado == "Sí" else "Página con sello detectado, sin logo detectado"
                })
        else:
            # Si no hay sellos, igual registramos la página con la info del logo
            rows.append({
                "Archivo": pdf_path.name,
                "Página": page_num,
                "Tipo_página": page_type,
                "Logo_detectado": logo_detectado,
                "Entidades_logo_detectadas": entidades_logo_detectadas_str, # Nueva columna aquí
                "Sello_detectado": "No",
                "Nombre_sello": "",
                "Cargo": "",
                "CIP": "",
                "Confianza_sello": "",
                "Rotación_OCR": "",
                "PSM_OCR": "",
                "Texto_OCR": "",
                "Recorte": "",
                "Validación_página": "Página con logo detectado, sin sello detectado" if logo_detectado == "Sí" else "Sin logo ni sello detectado"
            })

df_paginas = pd.DataFrame(rows)

print("\nVista previa:")
display(df_paginas.head(20))

# Guardar CSV de respaldo
csv_path = Path("outputs/VALIDACION_PAGINAS.csv")
df_paginas.to_csv(csv_path, index=False, encoding="utf-8-sig")

# Agregar hoja al Excel existente
with pd.ExcelWriter(REPORTE_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_paginas.to_excel(writer, sheet_name="VALIDACION_PAGINAS", index=False)

print("\nListo amiga.")
print(f"Hoja creada en: {REPORTE_XLSX}")
print(f"CSV respaldo: {csv_path}")
print(f"Recortes guardados en: {OUT_DIR}")

/content/ramec
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Clases del modelo:
{0: 'membrete', 1: 'codigo', 2: 'fecha', 3: 'contenido', 4: 'proyecto', 5: 'revisiones', 6: 'responsables', 7: 'validacion_profesional', 8: 'entidades', 9: 'validacion_profesional_hoja_control', 10: 'fecha_ultima_revision_hoja_control', 11: 'proyecto_caratula', 12: 'titulo_documento_caratula', 13: 'num_documento_caratula', 14: 'fecha_caratula', 15: 'num_documento_hoja_control', 16: 'titulo_documento_hoja_control', 17: 'fecha_aprobacion_hoja_control', 18: 'responsables_hoja_control', 19: 'firmas_aprobacion_paginas', 20: 'logo_entidades_caratula', 21: 'num_revision_hoja_control', 22: 'logo_entidades_paginas'}

Procesando: AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf

Procesando: AVP-SAV-9000-D-INF-CYT-ING-000001.pdf

Vista previa:


,Archivo,Página,Tipo_página,Logo_detectado,Entidades_logo_detectadas,Sello_detectado,Nombre_sello,Cargo,CIP,Confianza_sello,Rotación_OCR,PSM_OCR,Texto_OCR,Recorte,Validación_página
0,AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf,1,Carátula,Sí,"E\n1\n“e\ne,\nMinisterio\n|\nNILLO VIAL\ni\nMO...",No,,,,,,,,,"Página con logo detectado, sin sello detectado"
1,AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf,2,Hoja de control,Sí,"on\nme\n""e,\nMinisterio\n= ANILLO VIAL\nPROYEC...",No,,,,,,,,,"Página con logo detectado, sin sello detectado"
2,AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf,3,Contenido,Sí,= = ANILLO VIAL : SG PERU | Se rans\n== PROYEC...,Sí,Mayra Gómez Sandoval,JEFE,,0.909,90,12,MAYRA GÓMEZ SANDOVAL\nmeee eee eke\nDre eee ee...,outputs/validacion_paginas/AVP-SAV-3000-D-MDE-...,Página con logo y sello detectado
3,AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf,3,Contenido,Sí,= = ANILLO VIAL : SG PERU | Se rans\n== PROYEC...,Sí,Eliseo Alvarez Ralomares,ESPECIALISTA,2049,0.906,270,6,ELISEO ALVAREZ RALOMARES\n_\nE>\nESPECIALISTA\...,outputs/validacion_paginas/AVP-SAV-3000-D-MDE-...,Página con logo y sello detectado
4,AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf,3,Contenido,Sí,= = ANILLO VIAL : SG PERU | Se rans\n== PROYEC...,Sí,Ñez Gutiérrez,,,0.9,270,11,ma...\nHaro\n-\nMIGUE\nÑEZ GUTIÉRREZ\nERENTE D...,outputs/validacion_paginas/AVP-SAV-3000-D-MDE-...,Página con logo y sello detectado
5,AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf,3,Contenido,Sí,= = ANILLO VIAL : SG PERU | Se rans\n== PROYEC...,Sí,Armando Gon,GERENTE,264301,0.885,180,12,ARMANDO GON\nEZ GONZÁLEZ\nReg. CIP\n264301\nma...,outputs/validacion_paginas/AVP-SAV-3000-D-MDE-...,Página con logo y sello detectado
6,AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf,4,Contenido,Sí,= = ANILLO VIAL : SG PERU | Se rans\n== PROYEC...,Sí,Mayra Gómez Sandoval,JEFE,,0.909,90,12,MAYRA GÓMEZ SANDOVAL\nmeee eee eke\nDre eee ee...,outputs/validacion_paginas/AVP-SAV-3000-D-MDE-...,Página con logo y sello detectado
7,AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf,4,Contenido,Sí,= = ANILLO VIAL : SG PERU | Se rans\n== PROYEC...,Sí,Eliseo Alvarez Ralomares,ESPECIALISTA,2049,0.907,270,6,ELISEO ALVAREZ RALOMARES\n_\nE>\nESPECIALISTA\...,outputs/validacion_paginas/AVP-SAV-3000-D-MDE-...,Página con logo y sello detectado
8,AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf,4,Contenido,Sí,= = ANILLO VIAL : SG PERU | Se rans\n== PROYEC...,Sí,Ñez Gutiérrez,,,0.905,270,11,ma...\nHaro\n-\nMIGUE\nÑEZ GUTIÉRREZ\nERENTE D...,outputs/validacion_paginas/AVP-SAV-3000-D-MDE-...,Página con logo y sello detectado
9,AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf,4,Contenido,Sí,= = ANILLO VIAL : SG PERU | Se rans\n== PROYEC...,Sí,Armando Gon,GERENTE,264301,0.885,180,12,ARMANDO GON\nEZ GONZÁLEZ\nReg. CIP\n264301\nma...,outputs/validacion_paginas/AVP-SAV-3000-D-MDE-...,Página con logo y sello detectado



Listo amiga.
Hoja creada en: outputs/Reporte_validacion.xlsx
CSV respaldo: outputs/VALIDACION_PAGINAS.csv
Recortes guardados en: outputs/validacion_paginas


In [34]:
import re
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

REPORTE_XLSX = Path("outputs/Reporte_validacion.xlsx")

df = pd.read_excel(REPORTE_XLSX, sheet_name="VALIDACION_PAGINAS")

def limpiar(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    if x.lower() == "nan":
        return ""
    return x

def unir_unicos_desde_textos(valores):
    salida = []
    for v in valores:
        v = limpiar(v)
        if not v:
            continue

        partes = [p.strip() for p in v.split("|") if p.strip()]
        for p in partes:
            if p not in salida:
                salida.append(p)

    return " | ".join(salida)

def paginas_lista(subdf):
    pags = sorted(subdf["Página"].dropna().astype(int).unique().tolist())
    return ", ".join(map(str, pags))

resumen_rows = []
observaciones_rows = []

for archivo, g in df.groupby("Archivo"):
    g = g.copy()
    g["Página"] = g["Página"].astype(int)

    total_paginas = g["Página"].nunique()

    preliminares = g[g["Tipo_página"].isin(["Carátula", "Hoja de control"])]
    contenido = g[g["Tipo_página"] == "Contenido"]

    total_contenido = contenido["Página"].nunique()

    paginas_sin_logo = g[g["Logo_detectado"] != "Sí"]
    paginas_contenido_sin_sello = contenido[contenido["Sello_detectado"] != "Sí"]

    paginas_con_logo = g[g["Logo_detectado"] == "Sí"]["Página"].nunique()
    paginas_con_sello = contenido[contenido["Sello_detectado"] == "Sí"]["Página"].nunique()

    entidades = unir_unicos_desde_textos(g["Entidades_logo_detectadas"])
    firmantes = unir_unicos_desde_textos(g["Nombre_sello"])
    cargos = unir_unicos_desde_textos(g["Cargo"])
    cips = unir_unicos_desde_textos(g["CIP"])

    obs = []

    if len(paginas_sin_logo) > 0:
        obs.append(f"Páginas sin logo detectado: {paginas_lista(paginas_sin_logo)}")

    if len(paginas_contenido_sin_sello) > 0:
        obs.append(f"Páginas de contenido sin sello lateral detectado: {paginas_lista(paginas_contenido_sin_sello)}")

    if not entidades:
        obs.append("No se identificaron entidades mediante OCR de logos")

    if total_contenido > 0 and not firmantes:
        obs.append("No se identificaron firmantes en páginas de contenido")

    if total_contenido > 0 and not cips:
        obs.append("No se identificaron CIP en páginas de contenido")

    if len(obs) == 0:
        validacion_general = "Conforme"
        observacion_general = "El documento presenta logo y sellos detectados en las páginas evaluadas."
    else:
        validacion_general = "Observado"
        observacion_general = " | ".join(obs)

    porcentaje_logo = round((paginas_con_logo / total_paginas) * 100, 2) if total_paginas else 0
    porcentaje_sello = round((paginas_con_sello / total_contenido) * 100, 2) if total_contenido else 0

    resumen_rows.append({
        "Archivo": archivo,
        "Total_páginas_evaluadas": total_paginas,
        "Páginas_preliminares": preliminares["Página"].nunique(),
        "Páginas_contenido": total_contenido,
        "Logo_detectado_documento": "Sí" if paginas_con_logo > 0 else "No",
        "%_páginas_con_logo": porcentaje_logo,
        "Entidades_detectadas": entidades,
        "Sello_detectado_en_contenido": "Sí" if paginas_con_sello > 0 else "No",
        "%_páginas_con_sello": porcentaje_sello,
        "Cantidad_páginas_con_sello": paginas_con_sello,
        "Firmantes_detectados": firmantes,
        "Cargos_detectados": cargos,
        "CIP_detectados": cips,
        "Validación_general": validacion_general,
        "Observación_general": observacion_general
    })

    # Hoja de observaciones: solo páginas problemáticas
    problemas = g[
        (g["Validación_página"].astype(str).str.contains("Revisar", case=False, na=False)) |
        (g["Logo_detectado"] != "Sí") |
        ((g["Tipo_página"] == "Contenido") & (g["Sello_detectado"] != "Sí"))
    ].copy()

    for _, r in problemas.iterrows():
        observaciones_rows.append({
            "Archivo": archivo,
            "Página": int(r["Página"]),
            "Tipo_página": limpiar(r.get("Tipo_página", "")),
            "Logo_detectado": limpiar(r.get("Logo_detectado", "")),
            "Sello_detectado": limpiar(r.get("Sello_detectado", "")),
            "Entidades_detectadas": limpiar(r.get("Entidades_logo_detectadas", "")),
            "Firmantes_detectados": limpiar(r.get("Nombre_sello", "")),
            "Validación_página": limpiar(r.get("Validación_página", ""))
        })

df_resumen_archivo = pd.DataFrame(resumen_rows)
df_observaciones = pd.DataFrame(observaciones_rows)

if df_observaciones.empty:
    df_observaciones = pd.DataFrame([{
        "Archivo": "",
        "Página": "",
        "Tipo_página": "",
        "Logo_detectado": "",
        "Sello_detectado": "",
        "Entidades_detectadas": "",
        "Firmantes_detectados": "",
        "Validación_página": "No se identificaron observaciones en las páginas evaluadas."
    }])

with pd.ExcelWriter(REPORTE_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_resumen_archivo.to_excel(writer, sheet_name="RESUMEN_ARCHIVO_FINAL", index=False)
    df_observaciones.to_excel(writer, sheet_name="OBSERVACIONES_PAGINAS", index=False)

# Formato visual
wb = load_workbook(REPORTE_XLSX)

for ws_name in ["RESUMEN_ARCHIVO_FINAL", "OBSERVACIONES_PAGINAS"]:
    ws = wb[ws_name]

    fill = PatternFill("solid", fgColor="1F4E78")
    font = Font(color="FFFFFF", bold=True)

    for cell in ws[1]:
        cell.fill = fill
        cell.font = font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len = 0
        for cell in col:
            value = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(value))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 70)

    ws.freeze_panes = "A2"

wb.save(REPORTE_XLSX)

print("Listo amiga.")
print("Se crearon las hojas finales resumidas:")
print("- RESUMEN_ARCHIVO_FINAL")
print("- OBSERVACIONES_PAGINAS")
print(f"Archivo actualizado: {REPORTE_XLSX}")

display(df_resumen_archivo)

Listo amiga.
Se crearon las hojas finales resumidas:
- RESUMEN_ARCHIVO_FINAL
- OBSERVACIONES_PAGINAS
Archivo actualizado: outputs/Reporte_validacion.xlsx


,Archivo,Total_páginas_evaluadas,Páginas_preliminares,Páginas_contenido,Logo_detectado_documento,%_páginas_con_logo,Entidades_detectadas,Sello_detectado_en_contenido,%_páginas_con_sello,Cantidad_páginas_con_sello,Firmantes_detectados,Cargos_detectados,CIP_detectados,Validación_general,Observación_general
0,AVP-SAV-3000-D-MDE-ITS-ING-000001.pdf,10,2,8,Sí,100.0,"E 1 “e e, Ministerio | NILLO VIAL i MO de Tran...",Sí,100.0,8,Mayra Gómez Sandoval | Eliseo Alvarez Ralomare...,JEFE | ESPECIALISTA | GERENTE,2049.0 | 264301.0,Conforme,El documento presenta logo y sellos detectados...
1,AVP-SAV-9000-D-INF-CYT-ING-000001.pdf,10,2,8,Sí,100.0,,No,0.0,0,,,,Observado,Páginas de contenido sin sello lateral detecta...


In [51]:
%cd /content/ramec

from pathlib import Path
import pandas as pd
import re
import unicodedata
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

REPORTE_XLSX = Path("outputs/Reporte_validacion.xlsx")

# ============================================================
# FUNCIONES GENERALES
# ============================================================

def limpiar(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    if x.lower() in ["nan", "none", "null"]:
        return ""
    return x

def sin_tildes(x):
    x = limpiar(x)
    x = unicodedata.normalize("NFKD", x)
    return "".join(c for c in x if not unicodedata.combining(c))

def normalizar_nombre(x):
    x = sin_tildes(x).upper()
    x = re.sub(r"[^A-ZÑ ]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def limpiar_cip_texto(x):
    x = limpiar(x)
    partes = [p.strip() for p in x.split("|") if p.strip()]
    salida = []
    for p in partes:
        p = re.sub(r"\.0$", "", p)
        p = re.sub(r"[^0-9]", "", p)
        if p and p not in salida:
            salida.append(p)
    return " | ".join(salida)

def unir_unicos(valores):
    salida = []
    for v in valores:
        v = limpiar(v)
        if not v:
            continue
        partes = [p.strip() for p in v.split("|") if p.strip()]
        for p in partes:
            if p and p not in salida:
                salida.append(p)
    return " | ".join(salida)

def lista_paginas(subdf):
    if subdf.empty:
        return ""
    pags = sorted(subdf["Página"].dropna().astype(int).unique().tolist())
    return ", ".join(map(str, pags))

def ok_no(valor):
    v = limpiar(valor).upper()
    if v in ["SI", "SÍ", "OK", "CONFORME", "CUMPLE", "TRUE"]:
        return "OK"
    if v in ["NO", "OBSERVADO", "NO CUMPLE", "FALSE"]:
        return "NO"
    return ""

def detectar_columna_archivo(df):
    for c in df.columns:
        if limpiar(c).lower() == "archivo":
            return c
    for c in df.columns:
        if "archivo" in limpiar(c).lower():
            return c
    return None

def detectar_columna_cumple(df):
    posibles = ["CUMPLE", "Cumple", "VALIDACION", "Validación", "Resultado", "RESULTADO", "Estado", "ESTADO"]
    for p in posibles:
        for c in df.columns:
            if limpiar(c).lower() == p.lower():
                return c
    return None

def obtener_cumple_hoja(nombre_hoja):
    try:
        temp = pd.read_excel(REPORTE_XLSX, sheet_name=nombre_hoja)
        col_archivo = detectar_columna_archivo(temp)
        col_cumple = detectar_columna_cumple(temp)

        if col_archivo is None or col_cumple is None:
            return {}

        d = {}
        for _, r in temp.iterrows():
            archivo = limpiar(r.get(col_archivo, ""))
            if not archivo:
                continue
            d[archivo] = ok_no(r.get(col_cumple, ""))
        return d
    except Exception:
        return {}

def obtener_ruta_hoja(nombre_hoja):
    try:
        temp = pd.read_excel(REPORTE_XLSX, sheet_name=nombre_hoja)
        col_archivo = detectar_columna_archivo(temp)

        col_ruta = None
        for c in temp.columns:
            if limpiar(c).lower() == "ruta":
                col_ruta = c
                break

        if col_archivo is None or col_ruta is None:
            return {}

        d = {}
        for _, r in temp.iterrows():
            archivo = limpiar(r.get(col_archivo, ""))
            ruta = limpiar(r.get(col_ruta, ""))
            if archivo and ruta:
                d[archivo] = ruta
        return d
    except Exception:
        return {}

# ============================================================
# RESPONSABLES DE HOJA DE CONTROL
# Se intenta leer desde VALIDACION_PROFESIONAL si existe.
# ============================================================

def obtener_responsables_hoja_control():
    posibles_hojas = ["VALIDACION_PROFESIONAL", "CONTROL_CAMBIOS_DOC"]
    responsables_por_archivo = {}

    columnas_clave = [
        "elabor", "revis", "aprob", "responsable",
        "firma_elabor", "firma_revis", "firma_aprob"
    ]

    ignorar = {
        "SI", "SÍ", "NO", "OK", "TRUE", "FALSE",
        "CONFORME", "OBSERVADO", "CUMPLE", "NO CUMPLE",
        "COMPLETO", "INCOMPLETO"
    }

    for hoja in posibles_hojas:
        try:
            temp = pd.read_excel(REPORTE_XLSX, sheet_name=hoja)
        except Exception:
            continue

        col_archivo = detectar_columna_archivo(temp)
        if col_archivo is None:
            continue

        cols_resp = []
        for c in temp.columns:
            cl = sin_tildes(c).lower()
            if any(k in cl for k in columnas_clave):
                cols_resp.append(c)

        for _, r in temp.iterrows():
            archivo = limpiar(r.get(col_archivo, ""))
            if not archivo:
                continue

            nombres = []
            for c in cols_resp:
                val = limpiar(r.get(c, ""))
                if not val:
                    continue

                # Evita meter columnas que son SI/NO
                if normalizar_nombre(val) in ignorar:
                    continue

                # Divide si hay varios nombres
                partes = re.split(r"\||;|,|", val)
                for p in partes:
                    p = limpiar(p)
                    if not p:
                        continue
                    pn = normalizar_nombre(p)
                    if pn in ignorar:
                        continue
                    if len(pn.split()) >= 2 and p not in nombres:
                        nombres.append(p)

            if nombres:
                responsables_por_archivo[archivo] = " | ".join(nombres)

    return responsables_por_archivo

responsables_hoja_control = obtener_responsables_hoja_control()

# ============================================================
# LEER DETALLE PÁGINA POR PÁGINA
# ============================================================

df = pd.read_excel(REPORTE_XLSX, sheet_name="VALIDACION_PAGINAS")

df["Página"] = df["Página"].astype(int)

for col in [
    "Logo_detectado_OCR",
    "Sello_detectado",
    "Firmante_detectado",
    "Cargo_detectado",
    "CIP_detectado",
    "Entidades_logo_detectadas",
    "Texto_logo_extraido",
]:
    if col not in df.columns:
        df[col] = ""

# ============================================================
# VALIDACION_PAGINAS - 1 FILA POR ARCHIVO
# ============================================================

rows_paginas = []

for archivo, g in df.groupby("Archivo"):
    g = g.copy()

    contenido = g[g["Tipo_página"] == "Contenido"]

    total_contenido = contenido["Página"].nunique()
    paginas_con_sello = contenido[contenido["Sello_detectado"] == "Sí"]["Página"].nunique()
    paginas_sin_sello = contenido[contenido["Sello_detectado"] != "Sí"]

    porcentaje_sello = round((paginas_con_sello / total_contenido) * 100, 2) if total_contenido else 0

    firmantes = unir_unicos(g["Firmante_detectado"]) # Assuming 'Firmante_detectado' is an existing column or handled by previous fixes
    cargos = unir_unicos(g["Cargo_detectado"]) # Assuming 'Cargo_detectado' is an existing column or handled by previous fixes
    cips = limpiar_cip_texto(unir_unicos(g["CIP_detectado"])) # Assuming 'CIP_detectado' is an existing column or handled by previous fixes

    responsables = responsables_hoja_control.get(archivo, "")

    firmantes_norm = normalizar_nombre(firmantes)
    responsables_norm = normalizar_nombre(responsables)

    if responsables_norm and firmantes_norm:
        resp_lista = [normalizar_nombre(x) for x in responsables.split("|") if limpiar(x)]
        coincidencias = []

        for resp in resp_lista:
            palabras = [p for p in resp.split() if len(p) >= 4]
            if palabras and any(p in firmantes_norm for p in palabras):
                coincidencias.append(resp)

        firmantes_coinciden = "OK" if coincidencias else "NO"
    elif responsables_norm and not firmantes_norm:
        firmantes_coinciden = "NO"
    else:
        firmantes_coinciden = "NO" if not firmantes_norm else "SIN RESPONSABLES BASE"

    observaciones = []

    if total_contenido == 0:
        observaciones.append("No se identificaron páginas de contenido para evaluar sellos.")
    elif paginas_con_sello == total_contenido:
        observaciones.append("Se detectaron sellos laterales en todas las páginas de contenido evaluadas.")
    else:
        observaciones.append(f"Páginas de contenido sin sello lateral detectado: {lista_paginas(paginas_sin_sello)}.")

    if not firmantes:
        observaciones.append("No se identificaron firmantes en páginas de contenido.")

    if not cips:
        observaciones.append("No se identificaron CIP en páginas de contenido.")

    if responsables and firmantes_coinciden == "NO":
        observaciones.append("Los firmantes detectados no coinciden con los responsables registrados en la hoja de control.")

    if not responsables:
        observaciones.append("No se logró recuperar responsables desde la hoja de control para comparación automática.")

    cumple = "OK"
    if total_contenido == 0:
        cumple = "NO"
    if paginas_con_sello < total_contenido:
        cumple = "NO"
    if not firmantes:
        cumple = "NO"
    if not cips:
        cumple = "NO"
    if firmantes_coinciden == "NO":
        cumple = "NO"

    rows_paginas.append({
        "Archivo": archivo,
        "Sello_detectado_en_contenido": "Sí" if paginas_con_sello > 0 else "No",
        "%_páginas_con_sello": porcentaje_sello,
        "Cantidad_páginas_con_sello": paginas_con_sello,
        "Firmantes_detectados": firmantes,
        "Cargos_detectados": cargos,
        "Responsables_hoja_de_control": responsables,
        "Firmantes_coinciden": firmantes_coinciden,
        "CIP_detectados": cips,
        "CUMPLE": cumple,
        "Observación_general": " | ".join(observaciones)
    })

df_validacion_paginas = pd.DataFrame(rows_paginas)

# ============================================================
# VALIDACION_LOGOS - 1 FILA POR ARCHIVO
# ============================================================

ENTIDAD_ESPERADA = "PROINVERSIÓN"

def limpiar_entidades(entidades, texto_ocr):
    base = f"{limpiar(entidades)} {limpiar(texto_ocr)}"
    base_norm = sin_tildes(base).upper()

    salida = []

    if "PROINVERSION" in base_norm:
        salida.append("PROINVERSIÓN")

    if "MTC" in base_norm or "MINISTERIO" in base_norm or "TRANSPORTES" in base_norm:
        salida.append("MTC")

    if "OSITRAN" in base_norm:
        salida.append("OSITRAN")

    if "ANILLO VIAL" in base_norm or "CONCESIONARIA" in base_norm or "AVP" in base_norm:
        salida.append("Sociedad Concesionaria Anillo Vial")

    final = []
    for e in salida:
        if e not in final:
            final.append(e)

    return " | ".join(final)

rows_logos = []

for archivo, g in df.groupby("Archivo"):
    g = g.copy()

    total_paginas = g["Página"].nunique()
    paginas_con_logo = g[g["Logo_detectado_OCR"] == "Sí"]["Página"].nunique()
    porcentaje_logo = round((paginas_con_logo / total_paginas) * 100, 2) if total_paginas else 0

    paginas_sin_logo = g[g["Logo_detectado_OCR"] != "Sí"]

    entidades_raw = unir_unicos(g["Entidades_logo_detectadas"])
    texto_raw = unir_unicos(g["Texto_logo_extraido"])
    entidades_limpias = limpiar_entidades(entidades_raw, texto_raw)

    if not entidades_limpias and paginas_con_logo > 0:
        entidades_limpias = "Logo detectado; entidad no identificada por OCR"

    concedente_ok = "OK" if "PROINVERSIÓN" in entidades_limpias.upper() else "NO"

    observaciones = []

    if paginas_con_logo == total_paginas:
        observaciones.append("Se detectó logo en todas las páginas evaluadas.")
    else:
        observaciones.append(f"Páginas sin logo detectado: {lista_paginas(paginas_sin_logo)}.")

    if concedente_ok == "OK":
        observaciones.append("Se identificó PROINVERSIÓN como entidad esperada.")
    else:
        if "MTC" in entidades_limpias.upper():
            observaciones.append("Se detectó MTC, pero la entidad esperada como concedente es PROINVERSIÓN.")
        elif "Logo detectado" in entidades_limpias:
            observaciones.append("Se detectó presencia de logo, pero no se identificó PROINVERSIÓN mediante OCR.")
        else:
            observaciones.append("No se identificó PROINVERSIÓN mediante OCR de logos.")

    cumple = "OK" if paginas_con_logo > 0 and concedente_ok == "OK" else "NO"

    rows_logos.append({
        "Archivo": archivo,
        "Logo_detectado_documento": "Sí" if paginas_con_logo > 0 else "No",
        "%_páginas_con_logo": porcentaje_logo,
        "Entidad_esperada": ENTIDAD_ESPERADA,
        "Entidades_detectadas": entidades_limpias,
        "Concedente_OK": concedente_ok,
        "CUMPLE": cumple,
        "Observación_general": " | ".join(observaciones)
    })

df_validacion_logos = pd.DataFrame(rows_logos)

# ============================================================
# GENERAL
# ============================================================

hojas_validacion = [
    "ESTANDAR NOMENCLATURA",
    "COMPATIBILIDAD_DOCUMENTO",
    "COHERENCIA_DOCUMENTO",
    "CONTROL_CAMBIOS_DOC",
    "VALIDACION_PROFESIONAL",
]

cumples = {h: obtener_cumple_hoja(h) for h in hojas_validacion}

ruta_dict = {}
for h in hojas_validacion:
    ruta_dict.update(obtener_ruta_hoja(h))

archivos = sorted(set(df["Archivo"].astype(str)))

rows_general = []

for archivo in archivos:
    ruta = ruta_dict.get(archivo, f"pdfs/{archivo}")

    vp = df_validacion_paginas.loc[df_validacion_paginas["Archivo"] == archivo, "CUMPLE"]
    vl = df_validacion_logos.loc[df_validacion_logos["Archivo"] == archivo, "CUMPLE"]

    rows_general.append({
        "Ruta": ruta,
        "Archivo": archivo,
        "ESTANDAR NOMENCLATURA": cumples["ESTANDAR NOMENCLATURA"].get(archivo, ""),
        "COMPATIBILIDAD_DOCUMENTO": cumples["COMPATIBILIDAD_DOCUMENTO"].get(archivo, ""),
        "COHERENCIA_DOCUMENTO": cumples["COHERENCIA_DOCUMENTO"].get(archivo, ""),
        "CONTROL_CAMBIOS_DOC": cumples["CONTROL_CAMBIOS_DOC"].get(archivo, ""),
        "VALIDACION_PROFESIONAL": cumples["VALIDACION_PROFESIONAL"].get(archivo, ""),
        "VALIDACION_PAGINAS": vp.iloc[0] if not vp.empty else "",
        "VALIDACION_LOGOS": vl.iloc[0] if not vl.empty else "",
    })

df_general = pd.DataFrame(rows_general)

# ============================================================
# GUARDAR HOJAS
# ============================================================

with pd.ExcelWriter(REPORTE_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_general.to_excel(writer, sheet_name="GENERAL", index=False)
    # Changed sheet names to avoid overwriting detailed VALIDACION_PAGINAS and VALIDACION_LOGOS
    df_validacion_paginas.to_excel(writer, sheet_name="PJ3_VALIDACION_PAGINAS_RESUMEN", index=False)
    df_validacion_logos.to_excel(writer, sheet_name="PJ3_VALIDACION_LOGOS_RESUMEN", index=False)

# ============================================================
# FORMATO EXCEL
# ============================================================

wb = load_workbook(REPORTE_XLSX)

orden_hojas = [
    "GENERAL",
    "ESTANDAR NOMENCLATURA",
    "COMPATIBILIDAD_DOCUMENTO",
    "COHERENCIA_DOCUMENTO",
    "CONTROL_CAMBIOS_DOC",
    "VALIDACION_PROFESIONAL",
    "VALIDACION_PAGINAS",
    "VALIDACION_LOGOS",
    "VALIDACION_FINAL_PAGINAS",
    "DETALLE_SELLOS_PAGINAS",
    "DETALLE_LOGOS_PAGINAS",
    "RESUMEN_PAGINAS_LIMPIO",
    "RESUMEN_LOGOS_PAGINAS",
    "PJ3_VALIDACION_PAGINAS_RESUMEN",
    "PJ3_VALIDACION_LOGOS_RESUMEN",
    "AH5_VALIDACION_PAGINAS_RESUMEN",
    "AH5_VALIDACION_LOGOS_RESUMEN"
]

for nombre in reversed(orden_hojas):
    if nombre in wb.sheetnames:
        ws = wb[nombre]
        wb._sheets.remove(ws)
        wb._sheets.insert(0, ws)

azul = "1F4E78"
verde = "C6EFCE"
rojo = "FFC7CE"
amarillo = "FFF2CC"

thin = Side(style="thin", color="D9E2F3")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for ws in wb.worksheets:
    max_row = ws.max_row
    max_col = ws.max_column

    for cell in ws[1]:
        cell.fill = PatternFill("solid", fgColor=azul)
        cell.font = Font(color="FFFFFF", bold=True)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = border

    for row in ws.iter_rows(min_row=2, max_row=max_row, max_col=max_col):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = border

            val = limpiar(cell.value).upper()

            if val in ["OK", "SI", "SÍ", "CONFORME"]:
                cell.fill = PatternFill("solid", fgColor=verde)

            elif val in ["NO", "OBSERVADO"]:
                cell.fill = PatternFill("solid", fgColor=rojo)

            elif "REVISAR" in val or "NO APLICA" in val or "SIN RESPONSABLES" in val:
                cell.fill = PatternFill("solid", fgColor=amarillo)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len = 0
        for cell in col:
            value = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(value))

        if col_letter in ["A", "B"]:
            ws.column_dimensions[col_letter].width = min(max_len + 2, 48)
        else:
            ws.column_dimensions[col_letter].width = min(max_len + 2, 40)

    for r in range(1, max_row + 1):
        ws.row_dimensions[r].height = 35 if r == 1 else 45

# Ocultar hojas técnicas largas
for hoja in [
    "VALIDACION_FINAL_PAGINAS",
    "DETALLE_SELLOS_PAGINAS",
    "DETALLE_LOGOS_PAGINAS",
    "RESUMEN_PAGINAS_LIMPIO",
    "RESUMEN_LOGOS_PAGINAS",
]:
    if hoja in wb.sheetnames:
        wb[hoja].sheet_state = "hidden"

wb.save(REPORTE_XLSX)

print("Listo amiga.")
print("Excel final actualizado con:")
print("- GENERAL")
print("- VALIDACION_PAGINAS")
print("- VALIDACION_LOGOS")
print("Y las hojas técnicas quedaron ocultas.")
print(f"Archivo: {REPORTE_XLSX}")

display(df_general)
display(df_validacion_paginas)
display(df_validacion_logos)

/content/ramec


KeyError: 'Página'